In [ ]:
! pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 12.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime.

In [ ]:
import sys, os
import matplotlib
import time
import pandas as pd
import numpy
import ast
import json
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from sklearn.neighbors import KNeighborsClassifier as knnbase
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.naive_bayes import MultinomialNB as mnb
from sklearn.linear_model import LogisticRegression as LR
from sklearn.naive_bayes import GaussianNB as GNB

from autogluon.tabular import TabularDataset
from autogluon.tabular import TabularPredictor as task
from autogluon.core.utils import infer_problem_type

from sklearn.model_selection import GridSearchCV as GSCV
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler as MMS
from sklearn.preprocessing import StandardScaler as SS

from sklearn.metrics import accuracy_score, hamming_loss, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import multilabel_confusion_matrix as ML_matrix
from sklearn.metrics import precision_recall_fscore_support as score_multi
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from pickle import load, dump
from sklearn.utils import resample


In [ ]:
# -------------------------------- HELPERS ------------------------------------------ #
def split_df(Xdata, labels, testsplit=0.3):
	Xtrain,Xtest,ytrain,ytest = train_test_split(Xdata,labels,test_size=testsplit)
	return Xtrain, Xtest, ytrain, ytest

# Rescale values to fit in a range; default: 0-1
def normalize(Xtrain, Xtest):
	scaler = MMS(feature_range=(0,1))
	Xtrainscaled = scaler.fit_transform(Xtrain)
	Xtestscaled = scaler.transform(Xtest)
	return Xtrainscaled, Xtestscaled

# Scale values such that mean = 0, std dev. = 1; Ensures robustness for new data.
def standardize(Xtrain, Xtest):
	ss = SS()
	Xtrainscaled = ss.fit_transform(Xtrain)
	Xtestscaled = ss.transform(Xtest)
	return Xtrainscaled, Xtestscaled, ss

def micro_avg(y_test_multilabel, predictions):
	precision = precision_score(y_test_multilabel, predictions, average='micro')
	recall = recall_score(y_test_multilabel, predictions, average='micro')
	f1 = f1_score(y_test_multilabel, predictions, average='micro')

	print("::Micro-average::")
	print("Precision: {:.4f}, Recall: {:.4f}, F1-measure: {:.4f}".format(precision, recall, f1))
	print("\n\n")
	return precision, recall, f1

def macro_avg(y_test_multilabel, predictions):
	precision = precision_score(y_test_multilabel, predictions, average='macro')
	recall = recall_score(y_test_multilabel, predictions, average='macro')
	f1 = f1_score(y_test_multilabel, predictions, average='macro')

	print("\nMacro-average: ")
	print("Precision: {:.4f}, Recall: {:.4f}, F1-measure: {:.4f}".format(precision, recall, f1))
	return

def per_class_dist(ytest, ypred, classorder):
	perclass = classification_report(ytest, ypred)
	print("Per class classification report: ", perclass)
	precision, recall, fscore, support = score_multi(ytest, ypred, average="micro")
	print('micro-precision: {}'.format(precision))
	print('micro-recall: {}'.format(recall))
	print('micro-fscore: {}'.format(fscore))
	print('support: {}'.format(support))
	#print(classorder)
	return

def output_avg(total, ag_res1, ag_res2, fimp1, fimp2, auto_cmatrix, bestmodel, perf, auc_score, ff):
	print(auto_cmatrix)
	ff.write("-----------------Autogluon----------------\n")
	ff.write("Best model confusion matrix: \n")
	[tn,fp,fn,tp] = auto_cmatrix
	fpr = float(fp/(fp+tn)*100)
	ff.write("TN: "+str(tn)+" FP: "+str(fp)+" FN: "+str(fn)+" TP: "+str(tp)+"\n")
	ff.write("::Model performance on test data::\n")
	ff.write("AUC Score: "+str(auc_score)+"\n")
	ff.write("FPR: "+str(fpr)+"\n")
	ff.write("Best model: "+ bestmodel+" \n")
	ff.write("Performance summary: "+str(perf)+" \n")
	ff.write(str(ag_res1))
	if not fimp1 == None:
		ff.write("*Ft impo*\n")
		ff.write(str(fimp1.head(20))+"\n")
	ff.write("\n::Stacking & Weighted Ensembling of Models::\n")
	ff.write(str(ag_res2))
	if not fimp2 == None:
		ff.write("*Ft impo*\n")
		ff.write(str(fimp2.head(20))+"\n")
	ff.write("--------------------------------------------\n")
	ff.close()
	return

In [ ]:
def test_main(xtest, ytest, pred, testdf, traindf, calcftimpo=False):
	modelperf = pred.leaderboard(testdf, silent= True)
	print("[*]Model performance breakdown on Test data:")
	print(modelperf)
	ypred = pred.predict(xtest)
	ypredproba = pred.predict_proba(xtest)
	perf = pred.evaluate_predictions(y_true=ytest, y_pred=ypred, auxiliary_metrics= True)
	print("[*]Predictions: ", ypred)
	print("[*]Confidence in predictions:\n")
	print(pd.DataFrame(ypredproba, columns=pred.class_labels))
	# Each model score
	print("Perf: ", perf)
	print("Getting confusion matrix.....")
	cmatrix = confusion_matrix(ytest, ypred).ravel().tolist()
	print(cmatrix)
	auc_score = roc_auc_score(ytest, ypredproba.iloc[:, 1])
	print("AUC score for best model: ", auc_score)

	if calcftimpo:
		ftimpo = None
		ftimpo = pred.feature_importance(traindf)
		print("Feature Importance on test data: ", ftimpo)
	else:
		ftimpo = None
	bestmodel = pred.model_best
	bestmodel = pred.model_best

	# Find mistakes
	mistakes = xtest[ypred != ytest].copy()
	mistakes['true_label'] = ytest[ypred != ytest]
	mistakes['predicted_label'] = ypred[ypred != ytest]
	return modelperf, ftimpo, cmatrix, ypredproba, bestmodel, perf, auc_score, mistakes


def test_stack(xtest, ytest, predstack, testdf, traindf, calcftimpo=False):
	ypred = predstack.predict(xtest)
	ypredproba = predstack.predict_proba(xtest)
	perf = predstack.evaluate_predictions(y_true=ytest, y_pred=ypred, auxiliary_metrics= True)
	print("[*]Predictions: ", ypred)
	test_perf = predstack.leaderboard(testdf, silent=True)
	print("$$$$$$$$ RESULT STACKING $$$$$$$$\n", test_perf)
	ftimpo = None
	if calcftimpo:
		ftimpo = predstack.feature_importance(traindf)
		print("Feature Importance on test data: ", ftimpo)
	auc_score = roc_auc_score(ytest, ypredproba.iloc[:, 1])
	cmatrix = confusion_matrix(ytest, ypred).ravel().tolist()
	print("Confusion matrix stacked: ", cmatrix)
	print("AUC using stacked model: ", auc_score)
	return test_perf, ftimpo, cmatrix, auc_score

In [ ]:
def train_main(dataf, valdf, targetcol):
	agdir = os.getcwd()+'/AGmodels/'
	#dir = agdir+"/"+str(malinst)+"_"+str(hostfts)+"/"
	if not os.path.exists(agdir):
		os.system("mkdir "+agdir)

	predictor = task(label=targetcol, path=agdir, eval_metric='f1').fit( train_data=dataf, tuning_data=valdf, verbosity=3)
	return predictor

# Multi layer stacking takes predictions of base models and feeds to stack models
# AG will auto choose k= 10 fold cv, n=20 bagging repeats,
# L: 2 layers of models in stack followed by weighted-ensemble (higher weight for the model that performed well);
# Aggregate model predictions based on model weights and produce final prediction
def train_multilayerstacking(traindf, target):
	agdir_stack = os.getcwd()+'/AGmodels/stacked/' #+str(malinst)+"_"+str(hostfts)+"/"
	if not os.path.exists(agdir_stack):
		os.system("mkdir "+agdir_stack)
	predstack = task(label=target, path=agdir_stack, eval_metric='f1').fit(train_data= traindf, auto_stack=True, verbosity=3)
	return predstack


def main_ag(traindf, testdf, targetcol):
	# Displaying dataframe info
	x_test = testdf.iloc[:,:-1].copy()
	y_test = testdf.iloc[:,-1].copy()
	proxy_train = traindf[traindf['target'] == 1].shape
	proxy_test = testdf[testdf['target'] == 1].shape
	normal_train = traindf[traindf['target'] == 0].shape
	normal_test = testdf[testdf['target'] == 0].shape

	time.sleep(2)

  # Training binary classifiers: 8 base models, 2 DL models
	predictor = train_main(traindf, targetcol)
	predstack = train_multilayerstacking(traindf, targetcol)

	# Testing binary classifiers
	print("###################~Testing Trained Models (Never seen PCAPS)~############################")
	res1, fimp1, cmatrix, ypred_proba, perf, auc_score = test_main(x_test, y_test, predictor, testdf, traindf)
	# Uncomment for test results with feature importance (longer run time)
	##res1, fimp1, cmatrix, ytest, ypred_proba, bestmodel, perf, auc_score = test_main(Xtest, ytest, predictor, testdf, traindf, True)

	print("####################Stacking & Weighted Ensemble Testing###########################")
	res2, fimp2, cmatrixstacked, aucstacked = test_stack(x_test, y_test, predstack, testdf, traindf)
	# With feature importance
	##res2, fimp2, cmatrixstacked, aucstacked = test_stack(Xtest, ytest, predstack, testdf, traindf, True)

	return [res1, res2, fimp1, fimp2, cmatrix, perf, auc_score]

In [ ]:
foldtotal = 10
proxy_fts_path = '/content/features_lim_proxy_train_old.csv'
normal_fts_path = '/content/features_lim_normal_train_old.csv'
proxy_fts_test_path = '/content/features_lim_proxy_test_old.csv'
normal_fts_test_path = '/content/features_lim_normal_test_old.csv'
proxy_feats_val_path='/content/features_lim_proxy_val_old.csv'
normal_feats_val_path='/content/features_lim_normal_val_old.csv'

proxy_feats_train=pd.read_csv(proxy_fts_path)
normal_feats_train=pd.read_csv(normal_fts_path)

proxy_feats_test=pd.read_csv(proxy_fts_test_path)
normal_feats_test=pd.read_csv(normal_fts_test_path)

proxy_feats_val=pd.read_csv(proxy_feats_val_path)
proxy_feats_val['label'] = 1

normal_feats_val=pd.read_csv(normal_feats_val_path)
normal_feats_val['label'] = 0

normal_feats_train['label']=0 #50000
proxy_feats_train['label']=1 #5700
train=pd.concat([proxy_feats_train,normal_feats_train],ignore_index=True)
train=train.sample(frac=1, random_state=42)
train = train.reset_index(drop=True)
train_original = train.copy()
train=train.drop(columns=['pcap'])
train=train.drop(columns=['conn'])

val_data=pd.concat([proxy_feats_val,normal_feats_val])
val=val_data.sample(frac=1, random_state=42)
val = val.reset_index(drop=True)
val_original = val.copy()
val=val.drop(columns=['pcap'])
val=val.drop(columns=['conn'])


normal_feats_test['label']=0
proxy_feats_test['label']=1
test=pd.concat([proxy_feats_test,normal_feats_test],ignore_index=True)
test=test.sample(frac=1, random_state=42)
test = test.reset_index(drop=True)
test_original = test.copy()
test=test.drop(columns=['pcap'])
test=test.drop(columns=['conn'])

majority_class = train[train['label'] == 0]
minority_class = train[train['label'] == 1]
majority_downsampled = resample(majority_class,
                                replace=False, # sample without replacement
                                n_samples=len(minority_class), # match minority size
                                random_state=42)
train_balanced = pd.concat([majority_downsampled, minority_class])

In [ ]:
print(proxy_feats_train.shape)
print(normal_feats_train.shape)
print(proxy_feats_test.shape)
print(normal_feats_test.shape)
print(proxy_feats_val.shape)
print(normal_feats_val.shape)

(1933, 53)
(69219, 53)
(530, 53)
(12430, 53)
(617, 53)
(32993, 53)


In [ ]:
predictor = train_main(train_balanced, val, "label")


Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
GPU Count:          0
Memory Avail:       10.99 GB / 12.67 GB (86.7%)
Disk Space Avail:   73.99 GB / 107.72 GB (68.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='high'         : Strong accuracy w

[50]	valid_set's binary_logloss: 0.0711193	valid_set's f1: 0.856546
[100]	valid_set's binary_logloss: 0.0220677	valid_set's f1: 0.896652
[150]	valid_set's binary_logloss: 0.0123094	valid_set's f1: 0.91679
[200]	valid_set's binary_logloss: 0.00884057	valid_set's f1: 0.926426
[250]	valid_set's binary_logloss: 0.00734966	valid_set's f1: 0.934848
[300]	valid_set's binary_logloss: 0.00681296	valid_set's f1: 0.936978
[350]	valid_set's binary_logloss: 0.00651128	valid_set's f1: 0.939832
[400]	valid_set's binary_logloss: 0.00636778	valid_set's f1: 0.943425
[450]	valid_set's binary_logloss: 0.00640799	valid_set's f1: 0.944147
[500]	valid_set's binary_logloss: 0.00645778	valid_set's f1: 0.946319
[550]	valid_set's binary_logloss: 0.00640233	valid_set's f1: 0.946319
[600]	valid_set's binary_logloss: 0.00633982	valid_set's f1: 0.951426
[650]	valid_set's binary_logloss: 0.00635411	valid_set's f1: 0.949962
[700]	valid_set's binary_logloss: 0.00640509	valid_set's f1: 0.949962
[750]	valid_set's binary_

Saving /content/AGmodels/models/LightGBMXT/model.pkl
Saving /content/AGmodels/utils/attr/LightGBMXT/y_pred_proba_val.pkl
	0.9522	 = Validation score   (f1)
	23.81s	 = Training   runtime
	1.65s	 = Validation runtime
	20351.6	 = Inference  throughput (rows/s | 33610 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: LightGBM ...
	Fitting LightGBM with 'num_gpus': 0, 'num_cpus': 1
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05}


[50]	valid_set's binary_logloss: 0.0597073	valid_set's f1: 0.862334
[100]	valid_set's binary_logloss: 0.0137523	valid_set's f1: 0.926426
[150]	valid_set's binary_logloss: 0.00870394	valid_set's f1: 0.935557
[200]	valid_set's binary_logloss: 0.00797393	valid_set's f1: 0.936267
[250]	valid_set's binary_logloss: 0.0077329	valid_set's f1: 0.939117
[300]	valid_set's binary_logloss: 0.00745516	valid_set's f1: 0.943425
[350]	valid_set's binary_logloss: 0.00768753	valid_set's f1: 0.946319
[400]	valid_set's binary_logloss: 0.00773368	valid_set's f1: 0.946319
[450]	valid_set's binary_logloss: 0.00773717	valid_set's f1: 0.947773
[500]	valid_set's binary_logloss: 0.00775475	valid_set's f1: 0.948501
[550]	valid_set's binary_logloss: 0.0077823	valid_set's f1: 0.949231
[600]	valid_set's binary_logloss: 0.00777927	valid_set's f1: 0.949962
[650]	valid_set's binary_logloss: 0.00779693	valid_set's f1: 0.949962
[700]	valid_set's binary_logloss: 0.00779061	valid_set's f1: 0.949962
[750]	valid_set's binary_

Saving /content/AGmodels/models/LightGBM/model.pkl
Saving /content/AGmodels/utils/attr/LightGBM/y_pred_proba_val.pkl
	0.95	 = Validation score   (f1)
	22.45s	 = Training   runtime
	1.02s	 = Validation runtime
	32822.4	 = Inference  throughput (rows/s | 33610 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: RandomForestGini ...
	Fitting RandomForestGini with 'num_gpus': 0, 'num_cpus': 2
Saving /content/AGmodels/models/RandomForestGini/model.pkl
Saving /content/AGmodels/utils/attr/RandomForestGini/y_pred_proba_val.pkl
	0.9376	 = Validation score   (f1)
	2.03s	 = Training   runtime
	0.34s	 = Validation runtime
	98855.3	 = Inference  throughput (rows/s | 33610 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: RandomForestEntr ...
	Fitting RandomForestEntr with 'num_gpus': 0, 'num_cpus': 2
Saving /content/AGmodels/models/RandomForestEntr/model.pkl
Saving /content/AGmodels/utils/attr/RandomForestEntr/y_pred_proba_val.pkl
	0.9355	 = Validation score 

...

Saving /content/AGmodels/models/CatBoost/model.pkl
Saving /content/AGmodels/utils/attr/CatBoost/y_pred_proba_val.pkl
	0.9413	 = Validation score   (f1)
	92.39s	 = Training   runtime
	0.17s	 = Validation runtime
	194527.5	 = Inference  throughput (rows/s | 33610 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: ExtraTreesGini ...
	Fitting ExtraTreesGini with 'num_gpus': 0, 'num_cpus': 2
Saving /content/AGmodels/models/ExtraTreesGini/model.pkl
Saving /content/AGmodels/utils/attr/ExtraTreesGini/y_pred_proba_val.pkl
	0.955	 = Validation score   (f1)
	0.89s	 = Training   runtime
	0.43s	 = Validation runtime
	78250.2	 = Inference  throughput (rows/s | 33610 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: ExtraTreesEntr ...
	Fitting ExtraTreesEntr with 'num_gpus': 0, 'num_cpus': 2
Saving /content/AGmodels/models/ExtraTreesEntr/model.pkl
Saving /content/AGmodels/utils/attr/ExtraTreesEntr/y_pred_proba_val.pkl
	0.9505	 = Validation score   (f1)
	0.89s	

[0]	validation_0-logloss:0.60177	validation_0-_f1:-0.83220
[50]	validation_0-logloss:0.01842	validation_0-_f1:-0.89211
[100]	validation_0-logloss:0.01135	validation_0-_f1:-0.91611
[150]	validation_0-logloss:0.01053	validation_0-_f1:-0.91816
[200]	validation_0-logloss:0.01006	validation_0-_f1:-0.92021
[250]	validation_0-logloss:0.00970	validation_0-_f1:-0.92365
[300]	validation_0-logloss:0.00954	validation_0-_f1:-0.92573
[350]	validation_0-logloss:0.00941	validation_0-_f1:-0.92852
[400]	validation_0-logloss:0.00924	validation_0-_f1:-0.92981
[450]	validation_0-logloss:0.00918	validation_0-_f1:-0.92911
[500]	validation_0-logloss:0.00914	validation_0-_f1:-0.92981
[550]	validation_0-logloss:0.00910	validation_0-_f1:-0.93051
[600]	validation_0-logloss:0.00912	validation_0-_f1:-0.93122
[650]	validation_0-logloss:0.00910	validation_0-_f1:-0.93051
[700]	validation_0-logloss:0.00909	validation_0-_f1:-0.92981
[750]	validation_0-logloss:0.00910	validation_0-_f1:-0.92981
[800]	validation_0-logloss:

Saving /content/AGmodels/models/XGBoost/model.pkl
Saving /content/AGmodels/utils/attr/XGBoost/y_pred_proba_val.pkl
	0.9333	 = Validation score   (f1)
	69.13s	 = Training   runtime
	0.91s	 = Validation runtime
	36938.2	 = Inference  throughput (rows/s | 33610 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: NeuralNetTorch ...
	Fitting NeuralNetTorch with 'num_gpus': 0, 'num_cpus': 1
Tabular Neural Network treats features as the following types:
{
    "continuous": [
        "mean_vol_total_pkts",
        "mean_bytes_sent",
        "median_bytes_sent",
        "mode_bytes_sent",
        "75th_percentile_in",
        "75th_percentile_out",
        "75th_percentile_total",
        "nb_pkts_in",
        "nb_pkts_out",
        "nb_pkts_in_l30",
        "nb_pkts_out_l30",
        "std_pkt_conc_out20",
        "avg_pkt_conc_out20",
        "avg_order_in",
        "avg_order_out",
        "std_order_in",
        "std_order_out",
        "medconc",
        "maxconc",
      

[50]	valid_set's binary_logloss: 0.129706	valid_set's f1: 0.870922
[100]	valid_set's binary_logloss: 0.0397176	valid_set's f1: 0.859342
[150]	valid_set's binary_logloss: 0.0209714	valid_set's f1: 0.872159
[200]	valid_set's binary_logloss: 0.0185747	valid_set's f1: 0.874644
[250]	valid_set's binary_logloss: 0.0206231	valid_set's f1: 0.875892
[300]	valid_set's binary_logloss: 0.0224849	valid_set's f1: 0.881551
[350]	valid_set's binary_logloss: 0.0225604	valid_set's f1: 0.891794
[400]	valid_set's binary_logloss: 0.0213146	valid_set's f1: 0.898975
[450]	valid_set's binary_logloss: 0.0202297	valid_set's f1: 0.900954
[500]	valid_set's binary_logloss: 0.0198525	valid_set's f1: 0.904271
[550]	valid_set's binary_logloss: 0.0193771	valid_set's f1: 0.905605
[600]	valid_set's binary_logloss: 0.0192095	valid_set's f1: 0.90708
[650]	valid_set's binary_logloss: 0.0190065	valid_set's f1: 0.908419
[700]	valid_set's binary_logloss: 0.0188628	valid_set's f1: 0.908555
[750]	valid_set's binary_logloss: 0.0

Saving /content/AGmodels/models/LightGBMLarge/model.pkl
Saving /content/AGmodels/utils/attr/LightGBMLarge/y_pred_proba_val.pkl
	0.9126	 = Validation score   (f1)
	32.09s	 = Training   runtime
	3.49s	 = Validation runtime
	9628.0	 = Inference  throughput (rows/s | 33610 batch size)
Saving /content/AGmodels/models/trainer.pkl
Loading: /content/AGmodels/utils/attr/LightGBM/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/KNeighborsDist/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/XGBoost/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/NeuralNetTorch/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/RandomForestGini/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/RandomForestEntr/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/CatBoost/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/LightGBMXT/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/ExtraTreesGini/y_pred_proba_val.pkl
Loading: /content/AGmodels/

In [ ]:
x_test = test.iloc[:,:-1].copy()
y_test = test.iloc[:,-1].copy()
print("###################~Testing Trained Models ############################")
#res1, fimp1, cmatrix, ypred_proba, bestmodel, perf, auc_score = test_main(x_val, y_val, predictor, val_data, train)
# Uncomment for test results with feature importance (longer run time)
res1, fimp1, cmatrix, ypred_proba, bestmodel, perf, auc_score,mistakes = test_main(x_test, y_test, predictor, test, train, True)


Loading: /content/AGmodels/models/KNeighborsUnif/model.pkl


###################~Testing Trained Models ############################


Loading: /content/AGmodels/models/KNeighborsDist/model.pkl
Loading: /content/AGmodels/models/LightGBMXT/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/RandomForestGini/model.pkl
Loading: /content/AGmodels/models/RandomForestEntr/model.pkl
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/ExtraTreesGini/model.pkl
Loading: /content/AGmodels/models/ExtraTreesEntr/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model-internals.pkl
Loading: /content/AGmodels/models/XGBoost/model.pkl
Loading: /content/AGmodels/models/NeuralNetTorch/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/ExtraTreesEntr/model.pkl


[*]Model performance breakdown on Test data:
                  model  score_test  score_val eval_metric  pred_time_test  \
0   WeightedEnsemble_L2    0.970672   0.970889          f1        9.812239   
1              CatBoost    0.969416   0.941266          f1        0.274515   
2            LightGBMXT    0.968692   0.952160          f1        1.236927   
3              LightGBM    0.967742   0.949962          f1        0.746452   
4               XGBoost    0.960222   0.933333          f1        1.488205   
5      RandomForestEntr    0.959924   0.935459          f1        0.500352   
6      RandomForestGini    0.958015   0.937595          f1        0.377426   
7        ExtraTreesEntr    0.951830   0.950464          f1        0.660850   
8        ExtraTreesGini    0.948792   0.954969          f1        0.817651   
9       NeuralNetFastAI    0.943636   0.852006          f1        0.763002   
10        LightGBMLarge    0.942883   0.912593          f1        2.335763   
11       NeuralNetT

Loading: /content/AGmodels/models/ExtraTreesGini/model.pkl
Loading: /content/AGmodels/models/KNeighborsDist/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/LightGBMXT/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model-internals.pkl
Loading: /content/AGmodels/models/NeuralNetTorch/model.pkl
Loading: /content/AGmodels/models/RandomForestGini/model.pkl
Loading: /content/AGmodels/models/XGBoost/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/ExtraTreesEntr/model.pkl
Loading: /content/AGmodels/models/ExtraTreesGini/model.pkl
Loading: /content/AGmodels/models/KNeighborsDist/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: 

[*]Predictions:  0        0
1        0
2        1
3        0
4        0
        ..
12955    0
12956    0
12957    0
12958    0
12959    0
Name: label, Length: 12960, dtype: int64
[*]Confidence in predictions:

              0         1
0      0.999938  0.000062
1      0.963216  0.036784
2      0.000081  0.999919
3      0.999167  0.000833
4      0.989735  0.010265
...         ...       ...
12955  0.996564  0.003436
12956  0.999942  0.000058
12957  0.999166  0.000834
12958  0.998815  0.001185
12959  0.999921  0.000079

[12960 rows x 2 columns]
Perf:  {'f1': 0.9706717123935666, 'accuracy': 0.997608024691358, 'balanced_accuracy': 0.9833991104904446, 'mcc': 0.9694291338972953, 'precision': 0.9734345351043643, 'recall': 0.9679245283018868}
Getting confusion matrix.....
[12416, 14, 17, 513]
AUC score for best model:  0.9996750102460572


Loading: /content/AGmodels/models/ExtraTreesGini/model.pkl
Loading: /content/AGmodels/models/KNeighborsDist/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /content/AGmodels/models/LightGBMLarge/model.pkl
Loading: /content/AGmodels/models/LightGBMXT/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model.pkl
Loading: /content/AGmodels/models/NeuralNetFastAI/model-internals.pkl
Loading: /content/AGmodels/models/NeuralNetTorch/model.pkl
Loading: /content/AGmodels/models/RandomForestGini/model.pkl
Loading: /content/AGmodels/models/XGBoost/model.pkl
Loading: /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
	583.56s	= Expected runtime (116.71s per shuffle set)
Loading: /content/AGmodels/models/CatBoost/model.pkl
Loading: /content/AGmodels/models/ExtraTreesEntr/model.pkl
Loading: /content/AGmodels/models/ExtraTreesGini/model.pkl
Loading: /content/AGmodels/models/KNeighborsDist/model.pkl
Loading: /content/AGmodels/models/LightGBM/model.pkl
Loading: /con

Feature Importance on test data:                         importance    stddev   p_value  n  p99_high   p99_low
max_total                0.004110  0.006621  0.118722  5  0.017742 -0.009523
std_order_in             0.003406  0.002550  0.020241  5  0.008657 -0.001845
nb_pkts_in_f30           0.002790  0.001592  0.008626  5  0.006068 -0.000487
median_vol_total_pkts    0.002634  0.002934  0.057553  5  0.008675 -0.003407
gap_between_conns        0.002512  0.004188  0.125501  5  0.011135 -0.006112
75th_percentile_out      0.002098  0.001937  0.036301  5  0.006088 -0.001891
nb_pkts_in_l30           0.002074  0.001919  0.036509  5  0.006026 -0.001877
median_bytes_recv        0.000873  0.002887  0.267944  5  0.006818 -0.005071
nb_pkts_in               0.000716  0.001601  0.186950  5  0.004013 -0.002581
avg_pkt_conc_out20       0.000706  0.001578  0.186950  5  0.003956 -0.002544
mode_bytes_recv          0.000611  0.001365  0.186950  5  0.003422 -0.002201
mean_bytes_recv          0.000129  0.00734

In [ ]:
cmatrix

[8138, 18, 5, 2099]

In [ ]:
perf

{'f1': 0.9945510542525468,
 'accuracy': 0.9977582846003898,
 'balanced_accuracy': 0.9977083049731752,
 'mcc': 0.9931474073122498,
 'precision': 0.9914974019839395,
 'recall': 0.9976235741444867}

In [ ]:
mistakes

,pkts_rate,mean_total_pkts,median_total_pkts,mode_total_pkts,mean_bytes_sent,median_bytes_sent,mode_bytes_sent,mean_bytes_recv,median_bytes_recv,mode_bytes_recv,...,max_per_sec,maxconc,perc_in,perc_out,sum_altconc,sum_alt_per_sec,sum_number_pkts,sum_intertimestats,true_label,predicted_label
265,0.254080,251.88,52.0,40,301.080000,44.0,40,202.680000,60.0,40,...,1.0,11.0,0.50,0.50,20.0,49.0,100.0,301.037499,0,1
924,1.281268,381.38,49.5,40,408.520000,40.0,40,354.240000,60.0,40,...,1.0,11.0,0.50,0.50,21.0,49.0,100.0,127.491559,0,1
1885,2.065058,288.90,52.0,40,259.291667,40.0,40,316.230769,95.5,40,...,1.0,10.0,0.48,0.52,20.0,49.0,100.0,62.982319,0,1
1987,0.292995,248.56,52.0,40,149.875000,40.0,40,339.653846,79.0,40,...,1.0,10.0,0.48,0.52,20.0,49.0,100.0,277.691863,0,1
2015,1.011098,268.14,52.0,40,370.160000,263.0,40,166.120000,40.0,40,...,1.0,10.0,0.50,0.50,20.0,49.0,100.0,75.054863,1,0
2151,1.174370,404.56,52.0,40,329.640000,40.0,40,479.480000,200.0,40,...,1.0,11.0,0.50,0.50,21.0,49.0,100.0,149.763020,0,1
2162,0.536092,259.34,62.0,40,313.240000,44.0,40,205.440000,68.0,40,...,1.0,11.0,0.50,0.50,20.0,49.0,100.0,239.441295,0,1
2282,0.103912,223.14,64.0,40,264.296296,64.0,40,174.826087,68.0,40,...,1.0,11.0,0.54,0.46,19.0,49.0,100.0,576.297282,0,1
2411,6.021516,345.38,52.0,40,359.000000,40.0,40,331.760000,66.0,40,...,1.0,11.0,0.50,0.50,21.0,49.0,100.0,25.178886,1,0
2778,1.302797,190.04,65.5,40,96.307692,40.0,40,291.583333,89.5,40,...,1.0,10.0,0.52,0.48,20.0,49.0,100.0,129.103877,0,1


In [ ]:
selected_columns =  test_original.loc[mistakes.index, ['conn', 'pcap']]

# Save the selected rows and columns to a new DataFrame or file
selected_columns_copy = selected_columns.copy()  # Create a copy
selected_columns_copy.to_csv('selected_conn_pcap.csv', index=False)  # Save to CSV (optional)

print(selected_columns_copy)

                   conn           pcap
265     normal_conn_530  mixed_26_high
924    normal_conn_2506  mixed_18_high
1885   normal_conn_2045   mixed_26_low
1987   normal_conn_1087  mixed_23_high
2015      proxy_conn_34   mixed_26_low
2151   normal_conn_3148  mixed_18_high
2162    normal_conn_626  mixed_18_high
2282    normal_conn_535  mixed_26_high
2411      proxy_conn_65   mixed_39_low
2778   normal_conn_3090   mixed_26_low
2806    normal_conn_528  mixed_26_high
3163    normal_conn_173   mixed_5_high
3934      proxy_conn_98   mixed_27_low
5781      proxy_conn_97   mixed_13_low
6366   normal_conn_3281   mixed_18_low
6765   normal_conn_1566   mixed_26_low
6964   normal_conn_2522   mixed_18_low
7164   normal_conn_2507  mixed_18_high
7469   normal_conn_3264   mixed_26_low
8591    normal_conn_838   mixed_33_low
9192   normal_conn_3290   mixed_18_low
9847     proxy_conn_142   mixed_39_low
10048   normal_conn_529  mixed_26_high


In [ ]:

[ag_res1, ag_res2, fimp1, fimp2, cmatrix, perf, aucscore] = main_ag(train, test, "target")
ff = open("./BinaryTraining.score", "w+")
output_avg(foldtotal, ag_res1, ag_res2, fimp1, fimp2, cmatrix, perf, aucscore, ff)

In [ ]:
foldtotal = 10
gw_fts_low_path = '/content/features_lim_low_gw.csv'
gw_fts_high_path = '/content/features_lim_high_gw.csv'
gw_fts_medium_path = '/content/features_lim_medium_gw.csv'
normal_fts_medium_path = '/content/features_lim_medium_nrml.csv'
normal_fts_low_path = '/content/features_lim_low_nrml.csv'
normal_fts_high_path='/content/features_lim_high_nrml.csv'
gw_feats_eval_path='/content/features_lim_eval_gw.csv'
normal_feats_eval_path='/content/features_lim_eval_nrml.csv'

gw_feats_low=pd.read_csv(gw_fts_low_path)
gw_feats_medium=pd.read_csv(gw_fts_medium_path)
gw_feats_high=pd.read_csv(gw_fts_high_path)
normal_feats_low=pd.read_csv(normal_fts_low_path)
normal_feats_medium=pd.read_csv(normal_fts_medium_path)
normal_feats_high=pd.read_csv(normal_fts_high_path)
normal_feats = pd.concat([normal_feats_low, normal_feats_high,normal_feats_medium], ignore_index=True)
normal_feats = shuffle(normal_feats, random_state=42)
normal_feats.reset_index(drop=True, inplace=True)
normal_feats['label']=0
gw_feats = pd.concat([gw_feats_low, gw_feats_high,gw_feats_medium], ignore_index=True)
gw_feats = shuffle(gw_feats, random_state=42)
gw_feats.reset_index(drop=True, inplace=True)
gw_feats['label']=1
train=pd.concat([gw_feats,normal_feats],ignore_index=True)
train=train.drop(columns=['number'])
train=train.drop(columns=['conn'])
train=train.reset_index(drop=True)

gw_feats_eval=pd.read_csv(gw_feats_eval_path)
gw_feats_eval['label'] = 1
normal_feats_eval=pd.read_csv(normal_feats_eval_path)
normal_feats_eval['label'] = 0
test=pd.concat([gw_feats_eval,normal_feats_eval])
test=test.drop(columns=['number'])
test=test.drop(columns=['conn'])
test=test.reset_index(drop=True)